# Guía de Desarrollo de Herramientas (Tools)

Cómo crear, probar e integrar herramientas en el dashboard EGOFET.

**Concepto:** una herramienta es un script Python que el dashboard ejecuta sobre un archivo adjunto (HDF5, CSV, foto) con parámetros elegidos por el usuario. Devuelve **figuras interactivas (plotly)**, **tablas** y un **resumen**. El dashboard guarda todo como un *tool run* en la base de datos.

**Filosofía del dashboard:** visualizar primero. No se escriben archivos salvo que se indique; las gráficas son interactivas, no imágenes. Exportar es una acción aparte y bajo demanda.

**Flujo:**
1. Adjuntar un HDF5 a una memoria (carta References).
2. Pulsar "Apply tool" → elegir herramienta → configurar parámetros → Run.
3. La herramienta se ejecuta una vez; su resultado queda como *tool run* (receta + figuras + tablas).
4. El run aparece bajo la referencia (References) y se muestra en Analysis y Figures.

**Rutas importantes:**
- Herramientas: `src/dashboard_tools/*.py`
- Gestor de runs: `dashboard/tool_runs.py`
- Base de datos: `data/memories.db` (tablas `tool_runs` y `tool_run_items`)
- Plantillas: `dashboard/templates/*.yaml`

## 1. ¿Qué es una Tool?

Una clase que declara:
- `name`, `display_name`, `description`
- `input_ref_type` (`"hdf5"`, `"csv"`, `"photo"`, ...`)
- `measurement_mode` (`"transfer"`, `"output"` o `None`)
- `parameters` (qué pide al usuario)

e implementa `run(ref_path, params, workdir) -> ToolResult`.

`workdir` es un directorio temporal desechable por si la herramienta necesita archivos de trabajo; se borra al terminar. La mayoría no lo usa.

```python
from src.dashboard_tools.base import Tool, ToolResult, ToolTable, ParameterSchema
import plotly.graph_objects as go

class MiTool(Tool):
    name = "mi_tool"
    display_name = "Mi herramienta"
    description = "Hace algo útil"
    input_ref_type = "hdf5"
    parameters = ()

    def run(self, ref_path, params, workdir):
        fig = go.Figure(go.Scatter(x=[1, 2, 3], y=[1, 4, 9]))
        return ToolResult(figures=[fig], summary={"n": 3})
```

In [ ]:
import os, sys
from pathlib import Path

PROJECT_ROOT = Path(os.getcwd()).resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'src').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src import db
from src.dashboard_tools import list_tools, get_tool
from src.dashboard_tools.base import Tool, ToolResult, ToolTable, ParameterSchema

db.init_db()
print('Project root:', PROJECT_ROOT)
print('Database:', db.DB_PATH)
print('Herramientas:', [t.name for t in list_tools()])

## 2. Anatomía de una Tool

| Atributo | Tipo | Descripción |
|----------|------|-------------|
| `name` | `str` | id único, snake_case |
| `display_name` | `str` | nombre legible en la UI |
| `description` | `str` | explicación que aparece en el diálogo |
| `input_ref_type` | `str` | `hdf5`, `csv`, `photo`, `figure` |
| `measurement_mode` | `str \| None` | `transfer`, `output`, o `None` |
| `parameters` | `tuple[ParameterSchema]` | formulario del usuario |

**Método `run`:**
```python
def run(self, ref_path: Path, params: dict, workdir: Path) -> ToolResult:
    p = self.coerce(params)        # valida y tipa los parámetros
    ...                            # procesa ref_path
    return ToolResult(figures=[...], tables=[...], summary={...}, description="...")
```

## 3. Los 7 tipos de parámetros

`ParameterSchema(name, label, type, default, choices, help)`.

| Tipo | Widget | Ejemplo |
|------|--------|---------|
| `"float"` | número decimal | `v_min: -0.40` |
| `"int"` | número entero | `ncols: 3` |
| `"bool"` | switch | `loop: True` |
| `"str"` | texto | `title: "análisis"` |
| `"float_pair"` | dos números min/max | `v_min_range: (-0.40, -0.38)` |
| `"choice"` | desplegable | `color: "blue"` |
| `"measurements"` | checkboxes de mediciones | seleccionar del HDF5 |

```python
parameters = (
    ParameterSchema("measurements", "Mediciones", "measurements", (),
                    help="Vacío = todas las mediciones del archivo"),
    ParameterSchema("v_min_range", "v_min (V)", "float_pair", (-0.40, -0.38)),
    ParameterSchema("loop", "Sweep inverso", "bool", True),
)
```

El diálogo genera el formulario a partir de este esquema; no hay que tocar la UI para agregar parámetros.

## 4. ToolResult: qué devuelve una herramienta

```python
@dataclass
class ToolTable:
    label: str
    records: list      # lista de dicts (filas)

@dataclass
class ToolResult:
    figures: list      # go.Figure (plotly). Se guarda su spec, NO una imagen.
    tables: list       # ToolTable (datos, no CSV)
    summary: dict      # métricas cortas (median, IQR, ...)
    description: str   # frase humana para el notify
```

El dashboard guarda cada figura como su **spec de plotly** (JSON) en `tool_run_items.content_json`, y cada tabla como `{"records": [...]}`. No se escribe nada a disco.

```python
return ToolResult(
    figures=[fig],
    tables=[ToolTable(label="vth_results", records=df.to_dict("records"))],
    summary={"n_measurements": 23, "V_th (V)": {"median": 0.034}},
    description="V_th for 23 transfer curves",
)
```

## 5. Registro automático

`src/dashboard_tools/registry.py` descubre las herramientas con `pkgutil.iter_modules`:
1. Ignora módulos que empiezan con `_` (ej: `_common.py`), `base.py` y `registry.py`.
2. Importa cada módulo del paquete.
3. Registra cada subclase de `Tool` definida en él, bajo su `name`.

Para añadir una herramienta: crear `src/dashboard_tools/mi_tool.py`, definir la clase y reiniciar el dashboard.

```python
from src.dashboard_tools import list_tools, get_tool
for t in list_tools():
    print(t.name, t.display_name, t.input_ref_type)
```

In [ ]:
for tool in list_tools():
    print(f"{tool.name:16s} | {tool.display_name:34s} | {tool.input_ref_type} | mode={tool.measurement_mode}")
print()
t = get_tool('transfer_vth')
print('transfer_vth parameters:', [(p.name, p.type) for p in t.parameters])

## 6. El flujo completo

```
Apply tool (diálogo)          apply_tool()                 tool_runs + tool_run_items
  tool + params  ───────────▶  ejecuta la tool ──────────▶  (receta + figuras + tablas)
                                    │
                                    ▼
                        References (lista de tools aplicadas)
                        Analysis   (parámetros, resumen, tablas)
                        Figures    (figuras plotly interactivas)
```

`dashboard/tool_runs.py`:
- `apply_tool(memory_id, ref_path, tool_name, params) -> run_id`: valida, ejecuta con un `workdir` temporal, **reemplaza** el run previo del mismo `(memoria, ref, tool)` y guarda el resultado.
- `rerun_tool_run(run_id)`: vuelve a ejecutar con los parámetros guardados.
- `remove_tool_run(run_id)`: borra el run y sus items (cascada). Sin archivos.
- `summarize_run(run)`: frase para el notify.

## 7. Probar sin el dashboard

Puedes aplicar una herramienta directamente desde un notebook:

```python
from dashboard.tool_runs import apply_tool
import json

run_id = apply_tool(memory_id=1, ref_path="data/raw/x.hdf5",
                    tool_name="transfer_vth", params={})
run = db.get_tool_run(run_id)
print(run["tool_name"], [i["kind"] for i in run["items"]])
```

In [ ]:
import json
from dashboard.tool_runs import apply_tool, summarize_run, rerun_tool_run, remove_tool_run

# Buscar una memoria con un HDF5 adjunto
target_mem, target_ref = None, None
for m in db.list_memories():
    for r in db.get_memory(m['id'])['references']:
        if r['ref_type'] == 'hdf5':
            target_mem, target_ref = m, r['ref_path']
            break
    if target_mem:
        break

print('Memoria:', target_mem['title'] if target_mem else None)
print('Referencia:', target_ref)

In [ ]:
if target_mem:
    run_id = apply_tool(target_mem['id'], target_ref, 'transfer_vth', {})
    run = db.get_tool_run(run_id)
    print('Run:', run['tool_name'], '| items:', [(i['kind'], i['label']) for i in run['items']])
    print('Resumen:', summarize_run(run))
    print('Summary keys:', list(json.loads(run['summary_json']).keys()))

## 8. Ejemplo conceptual: una herramienta para CSV

Para leer CSV en vez de HDF5, basta cambiar `input_ref_type` y quitar `measurement_mode`:

```python
import pandas as pd, plotly.express as px

class CSVSummary(Tool):
    name = "csv_summary"
    display_name = "CSV summary"
    input_ref_type = "csv"
    measurement_mode = None
    parameters = (
        ParameterSchema("y_column", "Columna Y", "str", ""),
    )
    def run(self, ref_path, params, workdir):
        p = self.coerce(params)
        df = pd.read_csv(ref_path)
        fig = px.line(df, x=df.columns[0], y=p["y_column"] or df.columns[-1])
        return ToolResult(
            figures=[fig],
            tables=[ToolTable(label="data", records=df.head(50).to_dict("records"))],
            summary={"rows": len(df)},
            description=f"CSV with {len(df)} rows",
        )
```

**Nota:** hoy el botón "Apply tool" solo aparece en referencias HDF5. Para CSV habría que mostrar el botón según el `input_ref_type` de cada herramienta (pendiente de implementar).

---

## 9. Flujo legacy (DB + CSV), solo como referencia

Antes de las herramientas, un notebook creaba la memoria, procesaba el HDF5, guardaba un CSV y registraba el análisis a mano:

```python
memory_id = db.create_memory(...)
db.add_reference(memory_id, "hdf5", "data/raw/x.hdf5")
df.to_csv("data/processed/results.csv")
db.add_reference(memory_id, "csv", "data/processed/results.csv")
db.save_analysis_result(memory_id, "mi_analisis", params, results_csv_path)
```

Hoy el dashboard hace todo esto de forma gestionable con `apply_tool`. Las filas de `analysis_results` que quedan de ese flujo son solo de referencia histórica.